In [0]:
path="abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/"

In [0]:
source_folder            =path+"source_stream_folder"

schema_folder            =path+"schema_stream__folder"

checkpoint_folder       =path+"checkpoint_stream_folder"

target_folder            =path+"target_stream_folder"

In [0]:
dbutils.fs.mkdirs(source_folder)
dbutils.fs.mkdirs(checkpoint_folder)
dbutils.fs.mkdirs(schema_folder)
dbutils.fs.mkdirs(target_folder)

True

In [0]:

df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_folder)
    .option("header", True)
    .option("delimiter", ",")
    .load(source_folder))

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-8728819735364390>, line 8
      1 df = (spark.readStream
      2     .format("cloudFiles")
      3     .option("cloudFiles.format", "csv")
      4     .option("cloudFiles.schemaLocation", schema_folder)
      5     .option("header", True)
      6     .option("delimiter", ",")
      7     .option("path", source_folder)
----> 8     .load())

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    219         accessor=wrapper,
    220         module_name=module_name,
   (...)
    224         duration=time.perf_counter() - start
    225     )
    226     return res

File /databricks/spark/python/pyspark/sql/s

In [0]:
(df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_folder)
    .option("path", target_folder)
    .outputMode("append")
    .toTable("shivasai_catalog.bronze.person_details"))

### schema evolution and  mergeschema true

In [0]:

df = (spark.readStream                         # Reads data continuously/incrementally using Structured Streaming
    .format("cloudFiles")                      # Activates Databricks Auto Loader as the streaming data source
    .option("cloudFiles.format", "csv")        # Specifies the incoming file format as CSV
    .option("cloudFiles.schemaLocation", schema_folder)  # Stores inferred schema and schema evolution information
    .option("cloudFiles.inferColumnTypes", True) # Infers actual column data types instead of defaulting CSV columns to STRING
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") #Add New column arrives in source CSV
    .option("cloudFiles.schemaEvolutionMode", "rescue") #Rescue the data if schema is not matching
    .option("cloudFiles.schemaEvolutionMode","FailOnNewColumns") #Fail if new column arrives in source CSV
    .option("cloudFiles.maxFilesPerTrigger", 1) # Specifies the maximum number of files to be processed in each micro-batch
    .option("header", True)                    # Treats the first row of each CSV file as column headers
    .option("delimiter", ",")                  # Specifies comma as the column separator
    .load(source_folder)                       # Specifies the ADLS/storage folder from which Auto Loader reads files
)

(df.writeStream
    .format("delta")                         # Specifies Delta Lake as the target data format
    .option("checkpointLocation", checkpoint_folder)  # Stores streaming progress and metadata to recover after failures
    .option("path", target_folder)            # Specifies the storage location where Delta files will be written
    .outputMode("append")                     # Appends only newly processed records to the target
    .option("mergeSchema", True)              # Allows new columns in the DataFrame to be added to the Delta table schema
    .toTable("shivasai_catalog.bronze.person_details_mergeschema")  # Writes the streaming data into this Unity Catalog Delta table
)

In [0]:
%sql
select * from shivasai_catalog.bronze.person_details_mergeschema 

prsn_id,prsn_name,_rescued_data,age,deptno,salary
107,srinivas,null,null,10,2000
108,phani,null,null,10,3000
109,krishna,null,null,20,4000
107,srinivas,null,23,null,null
108,phani,null,24,null,null
109,krishna,null,25,null,null
101,srinivas,null,null,null,null
102,phani,null,null,null,null
103,krishna,null,null,null,null
104,srinivas,null,null,null,null
